# See-through 無料GPU実行ノートブック（Kaggle / Colab）

一枚絵をLive2D用のレイヤー分解PSDに変換する。ローカルGPUがVRAM 8GB未満のPC向けの**無料ルート**。

## 事前設定（重要）
- **Kaggle**: 右パネル Session options → Accelerator: **GPU T4 x2 か P100** / Internet: **ON**（無料枠: 週30時間）
- **Colab**: ランタイム → ランタイムのタイプを変更 → **T4 GPU**（無料枠は変動）

## 使い方
上から順にセルを実行 → 画像をアップロード → 分解実行 → zipをダウンロード。
初回はモデル重み（数GB）のダウンロードで10分以上かかる。**まとめて処理するのが枠の節約になる。**

> このノートブックはGPUの無い環境で作成したため実行未検証。
> エラーが出た場合はセルの出力メッセージを添えて相談してください。


In [ ]:
# 1) GPU環境の確認
import torch

if torch.cuda.is_available():
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    bf16 = torch.cuda.is_bf16_supported()
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {vram_gb:.1f} GB / bf16対応: {bf16}')
else:
    vram_gb, bf16 = 0, False
    print('GPUが見えていません。アクセラレータ設定(T4/P100)を確認して再起動してください')


In [ ]:
# 2) See-throughのセットアップ（環境のtorchはそのまま使う）
import os

WORK = '/content' if os.path.exists('/content') else '/kaggle/working'
os.chdir(WORK)
if not os.path.exists('see-through'):
    !git clone https://github.com/shitagaki-lab/see-through
os.chdir('see-through')
!pip install -q -r requirements.txt
!ln -sf common/assets assets
print('セットアップ完了')
# 実行時にtorch関連のエラーが出る場合のみ、本家指定のバージョンを入れる(時間がかかる):
# !pip install torch==2.8.0+cu128 torchvision==0.23.0+cu128 torchaudio==2.8.0+cu128 --index-url https://download.pytorch.org/whl/cu128


In [ ]:
# 3) 一枚絵のアップロード（PNG推奨・長辺1280px以上）
import os

INPUT_DIR = os.path.join(WORK, 'inputs')
os.makedirs(INPUT_DIR, exist_ok=True)
try:
    from google.colab import files  # Colabの場合はダイアログが開く
    up = files.upload()
    for name, data in up.items():
        with open(os.path.join(INPUT_DIR, name), 'wb') as f:
            f.write(data)
except ImportError:
    print(f'Kaggleの場合: 右のファイルパネルから {INPUT_DIR} にアップロード、')
    print('またはDatasetとして追加して INPUT_DIR をそのパスに変更してください')
print('入力ファイル:', os.listdir(INPUT_DIR))


In [ ]:
# 4) レイヤー分解の実行（VRAMとbf16対応から自動でプロファイル選択）
if bf16 and vram_gb >= 15:
    cmd = f'python inference/scripts/inference_psd.py --srcp {INPUT_DIR} --save_to_psd'
elif bf16 and vram_gb >= 10:
    cmd = f'python inference/scripts/inference_psd.py --srcp {INPUT_DIR} --save_to_psd --group_offload'
else:
    # T4/P100はbf16非対応のためNF4量子化版(約8GB)を使う
    cmd = f'python inference/scripts/inference_psd_quantized.py --srcp {INPUT_DIR} --save_to_psd'
print('実行:', cmd)
!{cmd}


In [ ]:
# 5) 出力PSDをzipにまとめてダウンロード
import glob, shutil, os

psds = glob.glob('workspace/layerdiff_output/**/*.psd', recursive=True)
print(f'PSD {len(psds)}件:', [os.path.basename(p) for p in psds])
zip_path = shutil.make_archive(os.path.join(WORK, 'psd_output'), 'zip',
                               'workspace/layerdiff_output')
try:
    from google.colab import files
    files.download(zip_path)
except ImportError:
    print(f'Kaggleの場合: 右パネルの {zip_path} をダウンロードしてください')


## 次の手順（ローカルPCで・GPU不要）

ダウンロードしたPSDを展開し、リポジトリの正規化スクリプトにかける:

```bash
# まずレイヤー名の確認(初回)
python scripts/normalize_psd.py inspect path/to/xxx.psd

# 一括正規化
python scripts/batch_decompose.py --normalize-only path/to/psd_dir/ --output output/
```

未分類レイヤーが出たら `configs/layer_mapping.yaml` にパターンを追記する。
以降は docs/03（Cubismテンプレート量産）へ。
